# CTC and WER: hands-on (optional)
**LLMs From Basics · Lesson A1.2**

Implement the CTC collapse rule, greedy decoding and the CTC loss (by brute force and by dynamic programming, checked against PyTorch), then score transcripts with word error rate.

This notebook is optional practice for the lesson. Each exercise has:
1. a cell with a `# TODO` for you to fill in,
2. a **check** cell with `assert`s. Run it; a ✓ means success.

Stuck? Solutions are at the very bottom. Try first, though: *struggling to recall is what makes it stick.*

Uses `numpy`, `matplotlib` and `torch` (all pre-installed on Colab).

In [ ]:
import itertools
import numpy as np
import matplotlib.pyplot as plt
import torch
print("ready, torch", torch.__version__)

## 1 · The collapse rule
CTC maps a frame-level path to text: **merge runs of repeated symbols, then drop blanks**. Implement `ctc_collapse(path, blank="_")` for a string (or list) of symbols; return a string.

In [ ]:
def ctc_collapse(path, blank="_"):
    # TODO: merge repeats first, then remove blanks
    raise NotImplementedError("Exercise 1")

**Check**

In [ ]:
assert ctc_collapse("hhe_ll_loo__") == "hello"
assert ctc_collapse("hhellloo") == "helo"        # no blank between the l's
assert ctc_collapse("_cc_aa_t_") == "cat"
assert ctc_collapse("bb_oo_okk") == "book"
assert ctc_collapse("_aa_pp_pll_e") == "apple"   # the quiz question
assert ctc_collapse("____") == ""
print("✓ exercise 1")

## 2 · Greedy decoding
A CTC model outputs a probability matrix `probs` of shape `(T, V)`: one distribution over the vocabulary per frame. Greedy decoding takes the most likely symbol per frame, then collapses. Implement `greedy_decode(probs, vocab)` where `vocab[0]` is the blank `"_"`.

In [ ]:
def greedy_decode(probs, vocab):
    # TODO: argmax per frame -> symbols -> ctc_collapse
    raise NotImplementedError("Exercise 2")

**Check**

In [ ]:
vocab = ["_", "c", "a", "t"]
P = np.array([[0.1, 0.8, 0.05, 0.05],
              [0.2, 0.7, 0.05, 0.05],
              [0.7, 0.1, 0.1, 0.1],
              [0.1, 0.1, 0.7, 0.1],
              [0.6, 0.1, 0.2, 0.1],
              [0.1, 0.05, 0.05, 0.8]])
assert greedy_decode(P, vocab) == "cat"
assert greedy_decode(np.eye(4)[[1, 0, 1, 2, 3, 3]], vocab) == "ccat"
print("✓ exercise 2")

## 3 · CTC probability by brute force, checked against PyTorch
$$P(y \mid x) = \sum_{a \in \mathcal{B}^{-1}(y)} \prod_{t=1}^{T} p_t(a_t)$$
For tiny problems we can literally enumerate all $V^T$ paths. Implement `ctc_prob_bruteforce(probs, target)` where `probs` is `(T, V)` with blank at index 0 and `target` is a list of label indices (no blanks). Use `itertools.product(range(V), repeat=T)`.

The check compares $-\log P$ with `torch.nn.CTCLoss`, which expects **log**-probabilities of shape `(T, batch, V)`.

In [ ]:
def ctc_prob_bruteforce(probs, target):
    T, V = probs.shape
    total = 0.0
    # TODO: loop over every path, keep those whose collapse equals target, add their probability
    raise NotImplementedError("Exercise 3")

**Check**

In [ ]:
# the lesson's worked example: 2 frames, symbols {blank, a}
P2 = np.array([[0.4, 0.6], [0.3, 0.7]])
assert np.isclose(ctc_prob_bruteforce(P2, [1]), 0.88)
assert np.isclose(ctc_prob_bruteforce(np.array([[0.5, 0.5], [0.2, 0.8]]), [1]), 0.9)   # the faded example

rng = np.random.default_rng(0)
ctc = torch.nn.CTCLoss(blank=0, reduction="sum")
for target in ([1, 2], [1, 1], [2, 1, 2]):
    logits = rng.normal(size=(6, 3))
    probs = np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)
    mine = -np.log(ctc_prob_bruteforce(probs, target))
    lp = torch.log_softmax(torch.tensor(logits), dim=1).unsqueeze(1)      # (T, 1, V)
    theirs = ctc(lp, torch.tensor([target]), torch.tensor([6]), torch.tensor([len(target)])).item()
    assert np.isclose(mine, theirs, atol=1e-6), (target, mine, theirs)
print("✓ exercise 3: brute force matches torch.nn.CTCLoss; it summed", 3 ** 6, "paths each time")

## 4 · The forward algorithm
Brute force needs $V^T$ terms, hopeless for real audio. The **forward algorithm** does it in $O(T \cdot |y|)$. Insert blanks around every label: `ext = [0, y1, 0, y2, 0, ..., yL, 0]` (length $2L+1$). Let `alpha[t, s]` be the total probability of all path prefixes up to frame `t` that end at position `s` of `ext`.

- Start: `alpha[0, 0] = p_0(blank)`, `alpha[0, 1] = p_0(y1)`.
- Step: `alpha[t, s] = (alpha[t-1, s] + alpha[t-1, s-1] + [alpha[t-1, s-2] if allowed]) * p_t(ext[s])`.
  Skipping from `s-2` is allowed only when `ext[s]` is not blank **and** `ext[s] != ext[s-2]` (you can't jump over the blank between two identical labels: that's the double-l rule!).
- Answer: `alpha[T-1, -1] + alpha[T-1, -2]` (end on the last label or the final blank).

In [ ]:
def ctc_forward(probs, target):
    T = probs.shape[0]
    ext = [0]
    for y in target:
        ext += [y, 0]
    alpha = np.zeros((T, len(ext)))
    # TODO: initialise row 0, fill rows 1..T-1, return the final sum
    raise NotImplementedError("Exercise 4")

**Check**

In [ ]:
rng = np.random.default_rng(1)
for target in ([1], [1, 2], [1, 1], [2, 1, 2], [1, 1, 1]):
    logits = rng.normal(size=(7, 3))
    probs = np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)
    assert np.isclose(ctc_forward(probs, target), ctc_prob_bruteforce(probs, target)), target
print("✓ exercise 4: dynamic programming = brute force")

## 5 · Word error rate
$$\text{WER} = \frac{S + D + I}{N}$$
The minimum number of substitutions, deletions and insertions is the **word-level edit distance** (Levenshtein). Implement `wer(ref, hyp)` for two strings (split on spaces) with the classic dynamic program: `d[i][j]` = edits to turn the first `i` reference words into the first `j` hypothesis words.

In [ ]:
def wer(ref, hyp):
    r, h = ref.split(), hyp.split()
    # TODO: fill the (len(r)+1) x (len(h)+1) table, return d[-1][-1] / len(r)
    raise NotImplementedError("Exercise 5")

**Check**

In [ ]:
assert np.isclose(wer("turn the kitchen lights off", "turn kitchen light off please"), 0.6)   # worked example
assert np.isclose(wer("please set a timer for ten minutes now", "please set timer for tan minutes now"), 0.25)
assert wer("hello world", "hello world") == 0
assert np.isclose(wer("yes", "oh yes yes indeed"), 3.0)       # insertions push WER above 100%
print("✓ exercise 5")

## Experiment

**CTC learns its own alignment.** Below, 12 frames of free logits (no audio at all) are trained with `CTCLoss` to spell "hello". Nobody tells it which frame is which letter, yet it settles on an alignment, with blanks where it needs them. Run it and look at the heatmap.

In [ ]:
vocab = ["_", "h", "e", "l", "o"]
target = torch.tensor([[1, 2, 3, 3, 4]])
torch.manual_seed(0)
logits = torch.zeros(12, 1, 5, requires_grad=True)
opt = torch.optim.Adam([logits], lr=0.1)
ctc_loss = torch.nn.CTCLoss(blank=0)
for step in range(300):
    loss = ctc_loss(logits.log_softmax(-1), target, torch.tensor([12]), torch.tensor([5]))
    opt.zero_grad(); loss.backward(); opt.step()
probs = logits.softmax(-1)[:, 0].detach().numpy()
print("final loss", round(loss.item(), 4), "| greedy decode:", greedy_decode(probs, vocab))
plt.figure(figsize=(8, 2.6))
plt.imshow(probs.T, aspect="auto", cmap="Blues")
plt.yticks(range(5), vocab); plt.xlabel("frame"); plt.title("learned per-frame probabilities")
plt.colorbar(); plt.show()

**Try:** shrink the number of frames to 6. CTC needs at least $|y| + (\text{number of repeated neighbours})$ frames: here 5 + 1 = 6. What happens at 5 frames?

## Recap
- Collapse = merge repeats, then drop blanks; blanks separate genuine double letters.
- CTC's $P(y \mid x)$ sums over every alignment; the forward algorithm computes it in $O(T \cdot |y|)$ and matches `torch.nn.CTCLoss`.
- WER is a word-level edit distance divided by the reference length.

Back to the course to finish the lesson and add the review cards.

---
# Solutions
Only peek after a real attempt.

### Exercise 1

In [ ]:
# Solution (copy into the exercise cell above)
def ctc_collapse(path, blank="_"):
    merged = [c for i, c in enumerate(path) if i == 0 or c != path[i - 1]]
    return "".join(c for c in merged if c != blank)

### Exercise 2

In [ ]:
# Solution (copy into the exercise cell above)
def greedy_decode(probs, vocab):
    path = [vocab[i] for i in np.argmax(probs, axis=1)]
    return ctc_collapse(path, blank=vocab[0])

### Exercise 3

In [ ]:
# Solution (copy into the exercise cell above)
def ctc_prob_bruteforce(probs, target):
    T, V = probs.shape
    total = 0.0
    for path in itertools.product(range(V), repeat=T):
        merged = [s for i, s in enumerate(path) if i == 0 or s != path[i - 1]]
        if [s for s in merged if s != 0] == list(target):
            total += np.prod(probs[np.arange(T), path])
    return total

### Exercise 4

In [ ]:
# Solution (copy into the exercise cell above)
def ctc_forward(probs, target):
    T = probs.shape[0]
    ext = [0]
    for y in target:
        ext += [y, 0]
    S = len(ext)
    alpha = np.zeros((T, S))
    alpha[0, 0] = probs[0, ext[0]]
    if S > 1:
        alpha[0, 1] = probs[0, ext[1]]
    for t in range(1, T):
        for s in range(S):
            a = alpha[t - 1, s]
            if s >= 1:
                a += alpha[t - 1, s - 1]
            if s >= 2 and ext[s] != 0 and ext[s] != ext[s - 2]:
                a += alpha[t - 1, s - 2]
            alpha[t, s] = a * probs[t, ext[s]]
    return alpha[T - 1, -1] + (alpha[T - 1, -2] if S > 1 else 0.0)

### Exercise 5

In [ ]:
# Solution (copy into the exercise cell above)
def wer(ref, hyp):
    r, h = ref.split(), hyp.split()
    d = np.zeros((len(r) + 1, len(h) + 1), dtype=int)
    d[:, 0] = np.arange(len(r) + 1)
    d[0, :] = np.arange(len(h) + 1)
    for i in range(1, len(r) + 1):
        for j in range(1, len(h) + 1):
            d[i, j] = min(d[i - 1, j] + 1,                       # deletion
                          d[i, j - 1] + 1,                       # insertion
                          d[i - 1, j - 1] + (r[i - 1] != h[j - 1]))  # substitution / match
    return d[-1, -1] / len(r)